# Full revision run (one notebook, run all cells in order)
**How to use:** Runtime > Run all. Approve the Google Drive permission popup when it appears. Keep this tab open.

Order: 1 config, 2 data functions, 3 detectors and metrics, 4 main loop (5 seeds), 5 summary tables, 6 efficiency,
7 validity checks, 8 environment and ensemble-detector cost, 9 extension (5 seeds: baselines, bit-depth, thresholds, cross-dataset),
10 paired tests and confusion counts. Results are saved as CSV files in the revision_results folder on your Drive.

# Revision pipeline: leak-free adversarial-sample detection on DReLAB
**Task.** Negatives = clean flows (benign + original malware). Positives = RL-perturbed flows (RF/WnD x DDQN/SARSA).
The IDS classifiers (RF, XGBoost) are trained on clean data only (benign vs original malware).
The detector never sees adversarial samples; its threshold is calibrated on held-out clean validation data.
**Leaks removed.** Unnamed: 0, Label, label and n_actions are dropped. No resampling with replacement.
**Split.** Position-based chunks (same relative position in every file lands in the same split), so an original flow
and its adversarial version fall in the same split (approximately; see the README note at the end).

In [ ]:
import os, time, pickle, warnings
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score
from xgboost import XGBClassifier
warnings.filterwarnings("ignore")

try:
    from google.colab import drive
except ImportError:
    drive = None                      # not running in Colab
if drive is not None:
    drive.mount("/content/drive")     # errors are shown, not hidden
    assert os.path.exists("/content/drive/MyDrive"), "Google Drive did not mount - re-run and click Allow"

# ---------------- CONFIG (edit here) ----------------
BASE = os.environ.get("DRELAB_BASE", "/content/drive/MyDrive/ColabNotebooks/DReLAB")
OUT = os.environ.get("REV_OUT", "/content/drive/MyDrive/ColabNotebooks/revision_results")
FAMILIES = {"CTU": ["Virut", "Neris", "Rbot", "Menti", "Murlo"],
            "BOTNET": ["Virut", "Neris", "Rbot", "Menti", "Murlo"],
            "CICIDS2018": ["Zeus_Ares"]}
ATTACKS = [("RF", "DDQN"), ("RF", "SARSA"), ("WnD", "DDQN"), ("WnD", "SARSA")]
SEEDS = list(range(int(os.environ.get("N_SEEDS", "5"))))
N_CHUNKS = int(os.environ.get("N_CHUNKS", "20"))       # position blocks
BUFFER = 0.02                                          # drop rows within 2% of a chunk edge
MAX_BENIGN = int(os.environ.get("MAX_BENIGN", "150000"))        # systematic (ordered) subsample of benign
MAX_TRAIN_PER_CLASS = int(os.environ.get("MAX_TRAIN", "40000"))
MAX_EVAL_PER_SET = int(os.environ.get("MAX_EVAL", "30000"))     # cap per val/test set (speed)
FAR_TARGET_PCT = 95                                    # threshold = 95th percentile of clean validation scores
BITS = 8                                               # quantization bits for the cleaned-up squeezer
DROP = {"Unnamed: 0", "Label", "label", "n_actions"}
os.makedirs(OUT, exist_ok=True)


In [ ]:
# ---------------- DATA ----------------
def with_pos(df):
    df = df.copy()
    df["_pos"] = (np.arange(len(df)) + 0.5) / len(df)
    return df

def load_dataset(ds):
    d = f"{BASE}/{ds}"
    benign = pd.read_csv(f"{d}/benign.csv")
    if len(benign) > MAX_BENIGN:                       # ordered subsample, keeps temporal structure
        benign = benign.iloc[np.linspace(0, len(benign) - 1, MAX_BENIGN).astype(int)]
    fams = FAMILIES[ds]
    orig = [with_pos(pd.read_csv(f"{d}/malicious/{f}.csv")) for f in fams]
    adv = {(o, a): [with_pos(pd.read_csv(f"{d}/{o}/{a}/{f}.csv")) for f in fams] for o, a in ATTACKS}
    benign = with_pos(benign)
    ref = [benign] + orig + [x for v in adv.values() for x in v]
    feats = [c for c in benign.columns
             if c not in DROP and c != "_pos"
             and all(c in r.columns for r in ref)
             and all(pd.api.types.is_numeric_dtype(r[c]) for r in ref)]
    def pool(frames):
        df = pd.concat(frames, ignore_index=True)
        X = df[feats].replace([np.inf, -np.inf], np.nan)
        X = X.fillna(X.median()).values.astype(np.float32)
        return X, df["_pos"].values
    data = {"benign": pool([benign]), "orig": pool(orig)}
    for k, v in adv.items():
        data[k] = pool(v)
    return data, feats

def split_roles(seed):
    rng = np.random.RandomState(seed)
    perm = rng.permutation(N_CHUNKS)
    n_tr, n_va = int(N_CHUNKS * 0.5), int(N_CHUNKS * 0.2)
    role = np.empty(N_CHUNKS, dtype=object)
    role[perm[:n_tr]] = "train"
    role[perm[n_tr:n_tr + n_va]] = "val"
    role[perm[n_tr + n_va:]] = "test"
    return role

def take(pool, role, split, rng, cap):
    X, pos = pool
    ch = np.minimum((pos * N_CHUNKS).astype(int), N_CHUNKS - 1)
    frac = pos * N_CHUNKS - ch
    m = (role[ch] == split) & (frac > BUFFER) & (frac < 1 - BUFFER)
    Xs = X[m]
    if len(Xs) > cap:
        Xs = Xs[rng.choice(len(Xs), cap, replace=False)]
    return Xs


In [ ]:
# ---------------- SQUEEZERS ----------------
def make_quant(Xtr, bits=BITS, q=(0.5, 99.5)):
    """True k-bit quantization: clip to the training [q_lo, q_hi] range, then 2^k levels per feature."""
    lo = np.percentile(Xtr, q[0], axis=0); hi = np.percentile(Xtr, q[1], axis=0)
    span = np.maximum(hi - lo, 1e-8); L = 2 ** bits - 1
    def sq(X):
        Z = np.clip((X - lo) / span, 0, 1)
        return (np.round(Z * L) / L * span + lo).astype(np.float32)
    return sq

def paper_sq(X):
    """The original paper code: round(x*256)/256, then a trailing median over 3 adjacent feature columns."""
    X1 = np.round(X * 256) / 256
    s1 = np.full_like(X1, np.nan); s1[:, 1:] = X1[:, :-1]
    s2 = np.full_like(X1, np.nan); s2[:, 2:] = X1[:, :-2]
    return np.nanmedian(np.stack([X1, s1, s2]), axis=0).astype(np.float32)

# ---------------- SCORES ----------------
def conf(p): return np.maximum(p, 1 - p)

def base_probs(X, rf, xgb):
    return rf.predict_proba(X)[:, 1], xgb.predict_proba(X)[:, 1]

def independent_scores(X, rf, xgb, rng):
    """Detectors that do not depend on the squeezer."""
    p_rf, p_xg = base_probs(X, rf, xgb)
    pn = rf.predict_proba(X + rng.normal(0, 0.01, X.shape).astype(np.float32))[:, 1]
    return {"Noise sensitivity (RF)": np.abs(p_rf - pn),
            "Ensemble disagreement only": np.abs(p_rf - p_xg)}

def squeeze_scores(X, rf, xgb, sq):
    p_rf, p_xg = base_probs(X, rf, xgb)
    Xs = sq(X)
    s_rf, s_xg = base_probs(Xs, rf, xgb)
    return {"Feature squeezing (RF)": np.abs(p_rf - s_rf),
            "Confidence drop (RF)": np.abs(conf(p_rf) - conf(s_rf)),
            "Squeeze + ensemble, Eq.11 variance": np.var(np.vstack([s_rf, s_xg]), axis=0),
            "Squeeze + ensemble, Eq.24 sum": np.abs(p_rf - s_rf) + np.abs(p_xg - s_xg)}

PROPOSED = ["Squeeze + ensemble, Eq.11 variance", "Squeeze + ensemble, Eq.24 sum"]
BASELINES = ["Noise sensitivity (RF)", "Feature squeezing (RF)", "Confidence drop (RF)"]

# ---------------- METRICS ----------------
def metrics(neg_val, neg_te, pos_te):
    thr = np.percentile(neg_val, FAR_TARGET_PCT)
    tp = int((pos_te > thr).sum()); fn = len(pos_te) - tp
    fp = int((neg_te > thr).sum()); tn = len(neg_te) - fp
    tpr, fpr = tp / len(pos_te), fp / len(neg_te)
    prec = tpr / (tpr + fpr + 1e-12)                  # precision at balanced prevalence
    f1 = 2 * prec * tpr / (prec + tpr + 1e-12)
    y = np.r_[np.zeros(len(neg_te)), np.ones(len(pos_te))]; s = np.r_[neg_te, pos_te]
    return dict(thr=thr, TPR=tpr, FPR=fpr, Precision_bal=prec, F1_bal=f1,
                AUROC=roc_auc_score(y, s), AUPRC=average_precision_score(y, s),
                TP=tp, FN=fn, FP=fp, TN=tn)


In [ ]:
# ---------------- MAIN LOOP ----------------
rows, clf_rows = [], []
cache = {}
for ds in FAMILIES:
    print("=" * 60, "\nLoading", ds)
    cache[ds], feats = load_dataset(ds)
    print(f"{ds}: {len(feats)} features ->", feats)
    for seed in SEEDS:
        t0 = time.time()
        rng = np.random.RandomState(seed)
        role = split_roles(seed)
        D = cache[ds]
        # --- IDS training data: clean only (benign=0, original malware=1), balanced WITHOUT replacement
        Xb, Xo = take(D["benign"], role, "train", rng, 10**9), take(D["orig"], role, "train", rng, 10**9)
        n = min(len(Xb), len(Xo), MAX_TRAIN_PER_CLASS)
        Xb, Xo = Xb[rng.choice(len(Xb), n, replace=False)], Xo[rng.choice(len(Xo), n, replace=False)]
        Xtr = np.vstack([Xb, Xo]); ytr = np.r_[np.zeros(n), np.ones(n)]
        scaler = StandardScaler().fit(Xtr)
        Xtr_s = scaler.transform(Xtr).astype(np.float32)
        rf = RandomForestClassifier(n_estimators=150, max_depth=20, n_jobs=-1, random_state=seed).fit(Xtr_s, ytr)
        xgb = XGBClassifier(n_estimators=150, max_depth=8, learning_rate=0.1, eval_metric="logloss",
                            random_state=seed).fit(Xtr_s, ytr)
        # --- clean validation (threshold) and clean test (negatives)
        sc = lambda X: scaler.transform(X).astype(np.float32)
        clean_val = sc(np.vstack([take(D["benign"], role, "val", rng, MAX_EVAL_PER_SET),
                                  take(D["orig"], role, "val", rng, MAX_EVAL_PER_SET)]))
        Xb_te = sc(take(D["benign"], role, "test", rng, MAX_EVAL_PER_SET))
        Xo_te = sc(take(D["orig"], role, "test", rng, MAX_EVAL_PER_SET))
        clean_te = np.vstack([Xb_te, Xo_te])
        adv_te = {k: sc(take(D[k], role, "test", rng, MAX_EVAL_PER_SET)) for k in ATTACKS}
        # --- IDS sanity: accuracy on clean test, evasion rate of each attack (predicted benign)
        acc = (np.r_[rf.predict(Xb_te) == 0, rf.predict(Xo_te) == 1]).mean()
        clf_rows.append(dict(dataset=ds, seed=seed, attack="clean", RF_acc_or_evasion=acc))
        for k, Xa in adv_te.items():
            clf_rows.append(dict(dataset=ds, seed=seed, attack="/".join(k),
                                 RF_acc_or_evasion=float((rf.predict(Xa) == 0).mean())))
        # --- scores
        sets = {"val": clean_val, "neg": clean_te}
        sets.update({"/".join(k): v for k, v in adv_te.items()})
        ind = {nm: independent_scores(X, rf, xgb, rng) for nm, X in sets.items()}
        squeezers = {"quant%d" % BITS: make_quant(Xtr_s), "paper-style": paper_sq}
        for sq_name, sq in squeezers.items():
            sqs = {nm: squeeze_scores(X, rf, xgb, sq) for nm, X in sets.items()}
            allsc = {nm: {**ind[nm], **sqs[nm]} for nm in sets}
            for method in allsc["val"]:
                if method in ind["val"] and sq_name != "quant%d" % BITS:
                    continue                                  # squeezer-independent: record once
                pos_names = ["/".join(k) for k in ATTACKS]
                for pn in pos_names + ["ALL"]:
                    pos = (np.concatenate([allsc[p][method] for p in pos_names]) if pn == "ALL"
                           else allsc[pn][method])
                    m = metrics(allsc["val"][method], allsc["neg"][method], pos)
                    rows.append(dict(dataset=ds, seed=seed, attack=pn, squeezer=(sq_name if method not in ind["val"] else "-"),
                                     method=method, **m))
        print(f"  {ds} seed {seed} done in {time.time() - t0:.0f}s  (IDS clean acc {acc:.4f})")
    # keep last fitted models for the efficiency table
    cache[ds]["_last"] = (rf, xgb, scaler, feats)

res = pd.DataFrame(rows); clf = pd.DataFrame(clf_rows)
res.to_csv(f"{OUT}/per_seed_results.csv", index=False); clf.to_csv(f"{OUT}/ids_sanity.csv", index=False)


In [ ]:
# ---------------- SUMMARY TABLES (mean +/- std over seeds) ----------------
def pm(x): return f"{np.mean(x):.4f} +/- {np.std(x, ddof=1) if len(x) > 1 else 0:.4f}"
agg = (res.groupby(["dataset", "attack", "squeezer", "method"])
          .agg(TPR=("TPR", pm), FPR=("FPR", pm), F1_bal=("F1_bal", pm), AUROC=("AUROC", pm), AUPRC=("AUPRC", pm))
          .reset_index())
agg.to_csv(f"{OUT}/summary_mean_std.csv", index=False)
pd.set_option("display.width", 250, "display.max_colwidth", 40, "display.max_rows", 500)
print("\n===== SUMMARY: all four attacks pooled =====")
print(agg[agg.attack == "ALL"].drop(columns="attack").to_string(index=False))
print("\n===== IDS sanity: clean accuracy, and evasion rate of each attack (fraction classified benign) =====")
print(clf.groupby(["dataset", "attack"])["RF_acc_or_evasion"].agg(["mean", "std"]).round(4))

# ---------------- SIGNIFICANCE (paired t-test on pooled AUROC across seeds) ----------------
print("\n===== Paired t-test on AUROC (pooled attacks): proposed vs best baseline =====")
for ds in FAMILIES:
    for sq_name in ["quant%d" % BITS, "paper-style"]:
        sub = res[(res.dataset == ds) & (res.attack == "ALL")]
        base = {b: sub[(sub.method == b) & ((sub.squeezer == sq_name) | (sub.squeezer == "-"))].sort_values("seed").AUROC.values
                for b in BASELINES}
        best = max(base, key=lambda b: base[b].mean())
        for p in PROPOSED:
            a = sub[(sub.method == p) & (sub.squeezer == sq_name)].sort_values("seed").AUROC.values
            if len(a) > 1:
                t, pv = stats.ttest_rel(a, base[best])
                print(f"{ds:10s} {sq_name:11s} {p:36s} vs {best:24s} dAUROC={a.mean() - base[best].mean():+.4f}  p={pv:.4f}")


In [ ]:
# ---------------- EFFICIENCY (lightweight claim) ----------------
print("\n===== Efficiency (last fitted models; ms per 1000 samples; 5000-sample batch) =====")
eff = []
try:
    import psutil
    proc = psutil.Process()
except Exception:
    proc = None
for ds in FAMILIES:
    rf, xgb, scaler, feats = cache[ds]["_last"]
    X = scaler.transform(cache[ds]["benign"][0][:5000]).astype(np.float32)
    sq = make_quant(X)
    rng = np.random.RandomState(0)
    def timeit(fn):
        fn(); t = time.perf_counter(); fn(); return (time.perf_counter() - t) / len(X) * 1000 * 1000
    eff.append(dict(dataset=ds,
                    ids_RF_only_ms=timeit(lambda: rf.predict_proba(X)),
                    noise_and_ensemble_ms=timeit(lambda: independent_scores(X, rf, xgb, rng)),
                    proposed_ms=timeit(lambda: squeeze_scores(X, rf, xgb, sq)),
                    RF_size_MB=len(pickle.dumps(rf)) / 1e6,
                    XGB_size_MB=len(xgb.get_booster().save_raw()) / 1e6,
                    RSS_MB=(proc.memory_info().rss / 1e6 if proc else float("nan"))))
eff = pd.DataFrame(eff).round(3)
eff.to_csv(f"{OUT}/efficiency.csv", index=False)
print(eff.to_string(index=False))
print("\nSaved to", OUT)


## 7. Validity checks (integer counts, flow with evading samples only)

In [ ]:
# ---- DIAGNOSTIC CELL: run AFTER the main loop, in the same Colab session ----
# 1) Are adversarial flows "format-different" (fractional counts) from original malware?
# 2) Does the best detector still work on adversarial flows that actually EVADE the IDS?
COUNT_COLS = ["TotPkts", "InBytes", "OutBytes", "TotBytes"]
print("===== 1. Share of NON-integer values in count columns (original malware vs adversarial) =====")
for ds in FAMILIES:
    feats = cache[ds]["_last"][3]
    cols = [c for c in COUNT_COLS if c in feats]
    idx = [feats.index(c) for c in cols]
    out = {}
    X = cache[ds]["orig"][0][:, idx]
    out["original"] = (np.abs(X - np.round(X)) > 1e-6).mean(axis=0).round(3)
    for k in ATTACKS:
        X = cache[ds][k][0][:, idx]
        out["/".join(k)] = (np.abs(X - np.round(X)) > 1e-6).mean(axis=0).round(3)
    print(ds, "columns:", cols)
    for nm, v in out.items():
        print(f"   {nm:10s}", list(v))

print("\n===== 2. Detection on ALL adversarial flows vs only those that EVADE the IDS (last seed's models) =====")
seed = SEEDS[-1]
role = split_roles(seed)
rows2 = []
for ds in FAMILIES:
    rf, xgb, scaler, feats = cache[ds]["_last"]
    D = cache[ds]; rng = np.random.RandomState(100 + seed)
    sc = lambda X: scaler.transform(X).astype(np.float32)
    clean_val = sc(np.vstack([take(D["benign"], role, "val", rng, MAX_EVAL_PER_SET),
                              take(D["orig"], role, "val", rng, MAX_EVAL_PER_SET)]))
    clean_te = sc(np.vstack([take(D["benign"], role, "test", rng, MAX_EVAL_PER_SET),
                             take(D["orig"], role, "test", rng, MAX_EVAL_PER_SET)]))
    adv = np.vstack([sc(take(D[k], role, "test", rng, MAX_EVAL_PER_SET)) for k in ATTACKS])
    evaded = rf.predict(adv) == 0
    # the quantizer must be re-fit on the same scaled training distribution used in the loop
    Xtr = sc(np.vstack([take(D["benign"], role, "train", rng, MAX_TRAIN_PER_CLASS),
                        take(D["orig"], role, "train", rng, MAX_TRAIN_PER_CLASS)]))
    sq = make_quant(Xtr)
    def S(X):
        d = {**independent_scores(X, rf, xgb, np.random.RandomState(0)), **squeeze_scores(X, rf, xgb, sq)}
        return d
    sv, sn, sa = S(clean_val), S(clean_te), S(adv)
    for method in ["Ensemble disagreement only", "Squeeze + ensemble, Eq.11 variance"]:
        for label, mask in [("all adversarial", np.ones(len(adv), bool)), ("evading only", evaded)]:
            if mask.sum() < 200:
                rows2.append(dict(dataset=ds, method=method, subset=label, n=int(mask.sum()), TPR=np.nan, AUROC=np.nan))
                continue
            m = metrics(sv[method], sn[method], sa[method][mask])
            rows2.append(dict(dataset=ds, method=method, subset=label, n=int(mask.sum()),
                              TPR=round(m["TPR"], 4), FPR=round(m["FPR"], 4), AUROC=round(m["AUROC"], 4)))
print(pd.DataFrame(rows2).to_string(index=False))


## 8. Environment and cost of the ensemble-only detector

In [ ]:
# ---- ENVIRONMENT + COST of the ensemble-only detector (for the paper's hardware description and Table 8) ----
import sys, platform, subprocess, tracemalloc, sklearn, xgboost
print("Python", sys.version.split()[0], "| scikit-learn", sklearn.__version__, "| xgboost", getattr(xgboost, "__version__", "n/a"))
print("CPU cores:", os.cpu_count(), "|", platform.processor() or "")
try: print(subprocess.run("lscpu | grep 'Model name'", shell=True, capture_output=True, text=True).stdout.strip())
except Exception: pass
cost = []
for ds in FAMILIES:
    rf, xgb, scaler, feats = cache[ds]["_last"]
    X = scaler.transform(cache[ds]["benign"][0][:5000]).astype(np.float32)
    def ms(fn):
        fn(); t = time.perf_counter(); fn(); return (time.perf_counter() - t) / len(X) * 1e6
    det = lambda: np.abs(rf.predict_proba(X)[:, 1] - xgb.predict_proba(X)[:, 1])
    tracemalloc.start(); det(); peak = tracemalloc.get_traced_memory()[1] / 1e6; tracemalloc.stop()
    cost.append(dict(dataset=ds, ensemble_detector_ms_per_1000=round(ms(det), 2), xgb_only_ms_per_1000=round(ms(lambda: xgb.predict_proba(X)), 2),
                     peak_extra_array_MB=round(peak, 1)))
cost = pd.DataFrame(cost); cost.to_csv(f"{OUT}/ensemble_detector_cost.csv", index=False)
print(cost.to_string(index=False))


## 9. Extension: baselines, bit-depth sweep, threshold sweep, cross-dataset transfer (5 seeds)

In [ ]:
# ---- EXTENSION CELL: run AFTER the main loop, in the same Colab session ----
# Adds: (1) threshold sensitivity, (2) squeezing bit-depth sweep, (3) feature-space baselines
#       (Mahalanobis, Isolation Forest), (4) cross-dataset transfer on the features all 3 datasets share.
from sklearn.ensemble import IsolationForest
EXT_SEEDS = SEEDS
FEATS = {ds: cache[ds]["_last"][3] for ds in FAMILIES}
COMMON = [c for c in FEATS["CICIDS2018"] if all(c in FEATS[d] for d in FAMILIES)]
print("common features (%d):" % len(COMMON), COMMON)

def split_sets(ds, seed, cols):
    idx = [FEATS[ds].index(c) for c in cols]
    role = split_roles(seed); rng = np.random.RandomState(seed); D = cache[ds]
    g = lambda k, sp, cap: take(D[k], role, sp, rng, cap)[:, idx]
    Xb, Xo = g("benign", "train", 10**9), g("orig", "train", 10**9)
    n = min(len(Xb), len(Xo), MAX_TRAIN_PER_CLASS)
    Xb, Xo = Xb[rng.choice(len(Xb), n, replace=False)], Xo[rng.choice(len(Xo), n, replace=False)]
    return dict(train=(np.vstack([Xb, Xo]), np.r_[np.zeros(n), np.ones(n)]),
                val=np.vstack([g("benign", "val", MAX_EVAL_PER_SET), g("orig", "val", MAX_EVAL_PER_SET)]),
                neg=np.vstack([g("benign", "test", MAX_EVAL_PER_SET), g("orig", "test", MAX_EVAL_PER_SET)]),
                adv=np.vstack([g(k, "test", MAX_EVAL_PER_SET) for k in ATTACKS]))

def fit_models(Xtr, ytr, seed):
    sc = StandardScaler().fit(Xtr); Z = sc.transform(Xtr).astype(np.float32)
    rf = RandomForestClassifier(150, max_depth=20, n_jobs=-1, random_state=seed).fit(Z, ytr)
    xg = XGBClassifier(n_estimators=150, max_depth=8, learning_rate=0.1, eval_metric="logloss",
                       random_state=seed).fit(Z, ytr)
    return sc, Z, rf, xg

ens = lambda X, rf, xg: np.abs(rf.predict_proba(X)[:, 1] - xg.predict_proba(X)[:, 1])
rows_ext = []
for seed in EXT_SEEDS:
    common = {ds: split_sets(ds, seed, COMMON) for ds in FAMILIES}
    for ds in FAMILIES:
        S = split_sets(ds, seed, FEATS[ds])
        sc, Z, rf, xg = fit_models(*S["train"], seed)
        T = lambda X, sc=sc: sc.transform(X).astype(np.float32)
        val, neg, adv = T(S["val"]), T(S["neg"]), T(S["adv"])
        mu = Z.mean(0); P = np.linalg.pinv(np.cov(Z, rowvar=False) + 1e-3 * np.eye(Z.shape[1]))
        iso = IsolationForest(n_estimators=100, random_state=seed, n_jobs=-1).fit(Z[:20000])
        det = {"Ensemble disagreement only": lambda X: ens(X, rf, xg),
               "Mahalanobis (clean features)": lambda X: (((X - mu) @ P) * (X - mu)).sum(1),
               "Isolation Forest (clean features)": lambda X: -iso.score_samples(X)}
        for b in [2, 4, 6, 8]:
            det["Squeeze Eq.11, %d-bit" % b] = (lambda sq: lambda X: squeeze_scores(
                X, rf, xg, sq)["Squeeze + ensemble, Eq.11 variance"])(make_quant(Z, bits=b))
        for name, f in det.items():
            sv, sn, sa = f(val), f(neg), f(adv)
            m = metrics(sv, sn, sa)
            rows_ext.append(dict(exp="methods", dataset=ds, seed=seed, method=name,
                                 TPR=m["TPR"], FPR=m["FPR"], AUROC=m["AUROC"], AUPRC=m["AUPRC"]))
            if name == "Ensemble disagreement only":
                for tn, thr in [("pct90", np.percentile(sv, 90)), ("pct95", np.percentile(sv, 95)),
                                ("pct97.5", np.percentile(sv, 97.5)), ("pct99", np.percentile(sv, 99)),
                                ("mean+2sd", sv.mean() + 2 * sv.std()), ("mean+3sd", sv.mean() + 3 * sv.std())]:
                    rows_ext.append(dict(exp="threshold sweep", dataset=ds, seed=seed, method=tn,
                                         TPR=(sa > thr).mean(), FPR=(sn > thr).mean()))
        print(f"  methods/thresholds: {ds} seed {seed} done")
    for src in FAMILIES:                                  # cross-dataset transfer (common features only)
        S = common[src]; sc, Z, rf, xg = fit_models(*S["train"], seed)
        T = lambda X, sc=sc: sc.transform(X).astype(np.float32)
        thr = np.percentile(ens(T(S["val"]), rf, xg), FAR_TARGET_PCT)      # threshold from SOURCE clean data
        for tgt in FAMILIES:
            sn = ens(T(common[tgt]["neg"]), rf, xg); sa = ens(T(common[tgt]["adv"]), rf, xg)
            y = np.r_[np.zeros(len(sn)), np.ones(len(sa))]; s = np.r_[sn, sa]
            rows_ext.append(dict(exp="cross-dataset", dataset=f"{src}->{tgt}", seed=seed,
                                 method="Ensemble disagreement only", TPR=(sa > thr).mean(),
                                 FPR=(sn > thr).mean(), AUROC=roc_auc_score(y, s),
                                 AUPRC=average_precision_score(y, s)))
        print(f"  cross-dataset: source {src} seed {seed} done")

ext = pd.DataFrame(rows_ext); ext.to_csv(f"{OUT}/extension_results.csv", index=False)
pm4 = lambda x: f"{np.mean(x):.3f}+/-{np.std(x, ddof=1) if len(x) > 1 else 0:.3f}"
summ = ext.groupby(["exp", "dataset", "method"]).agg(TPR=("TPR", pm4), FPR=("FPR", pm4),
        AUROC=("AUROC", pm4), AUPRC=("AUPRC", pm4)).reset_index()
summ.to_csv(f"{OUT}/extension_summary.csv", index=False)
pd.set_option("display.width", 250, "display.max_rows", 500)
for e in ["methods", "threshold sweep", "cross-dataset"]:
    print(f"\n===== {e.upper()} =====")
    print(summ[summ.exp == e].drop(columns="exp").to_string(index=False))


## 10. Paired tests and confusion counts

In [ ]:
# ---- FINAL RESULTS CELL: paired tests + confusion counts. Reads the saved CSVs (run after the extension re-run). ----
import numpy as np, pandas as pd
from scipy import stats
OUT = globals().get("OUT", "/content/drive/MyDrive/ColabNotebooks/revision_results")
main = pd.read_csv(f"{OUT}/per_seed_results.csv")
ext = pd.read_csv(f"{OUT}/extension_results.csv")
ext = ext[ext.exp == "methods"]

def get(df, ds, method, squeezer=None, pooled=True):
    d = df[(df.dataset == ds) & (df.method == method)]
    if pooled and "attack" in d.columns: d = d[d.attack == "ALL"]
    if squeezer is not None: d = d[d.squeezer == squeezer]
    return d.sort_values("seed").set_index("seed")["AUROC"]

def compare(a, b):
    idx = a.index.intersection(b.index); a, b = a[idx], b[idx]
    diff = a - b
    t, p = stats.ttest_rel(a, b) if len(idx) > 1 else (np.nan, np.nan)
    return dict(n_seeds=len(idx), AUROC_ens=round(a.mean(), 3), AUROC_other=round(b.mean(), 3),
                mean_diff=round(diff.mean(), 3), t=round(t, 2), p=round(p, 4), ens_wins=f"{int((diff > 0).sum())}/{len(idx)}")

rows = []
for ds in ["CTU", "BOTNET", "CICIDS2018"]:
    ens = get(main, ds, "Ensemble disagreement only")
    for nm, sq in [("Noise sensitivity (RF)", "-"), ("Feature squeezing (RF)", "quant8"), ("Confidence drop (RF)", "quant8")]:
        rows.append(dict(dataset=ds, vs=nm, **compare(ens, get(main, ds, nm, sq))))
    ens_x = get(ext, ds, "Ensemble disagreement only", pooled=False)
    for nm in ["Mahalanobis (clean features)", "Isolation Forest (clean features)"]:
        rows.append(dict(dataset=ds, vs=nm, **compare(ens_x, get(ext, ds, nm, pooled=False))))
tests = pd.DataFrame(rows)
tests.to_csv(f"{OUT}/paired_tests_ensemble_vs_baselines.csv", index=False)
pd.set_option("display.width", 220, "display.max_rows", 200)
print("===== Paired t-tests on AUROC: ensemble disagreement vs each baseline =====")
print(tests.to_string(index=False))
print("(Seeds share the same underlying flows, so p-values are indicative, not formal.)")

# confusion counts, summed over seeds
cm_cols = ["TP", "FN", "FP", "TN"]
pooled = main[main.attack == "ALL"].groupby(["dataset", "squeezer", "method"])[cm_cols].sum().reset_index()
per_attack = main[main.attack != "ALL"].groupby(["dataset", "attack", "squeezer", "method"])[cm_cols].sum().reset_index()
pooled.to_csv(f"{OUT}/confusion_counts_pooled.csv", index=False)
per_attack.to_csv(f"{OUT}/confusion_counts_per_attack.csv", index=False)
print("\n===== Confusion counts summed over seeds (pooled attacks): proposed detector and 8-bit squeezing =====")
keep = pooled[pooled.method.isin(["Ensemble disagreement only", "Squeeze + ensemble, Eq.11 variance", "Squeeze + ensemble, Eq.24 sum",
                                  "Feature squeezing (RF)"]) & pooled.squeezer.isin(["-", "quant8"])]
print(keep.to_string(index=False))
print("\nSaved: paired_tests_ensemble_vs_baselines.csv, confusion_counts_pooled.csv, confusion_counts_per_attack.csv in", OUT)
